# 06 — pLLM zero-shot benchmark: ESM-ladder baseline (TEM-1 / Firnberg)

**Task.** Predict functional vs non-functional (`DMS_score_bin`) from **zero-shot** ESM-family
substitution scores alone — no training on the labels. This is the pLLM-score bounding baseline
(project decision D027), the counterpart to the AA-identity supervised benchmark in `02`.

**Why this notebook exists.** `02` established that amino-acid identity alone is predictive but
**position-bound**: it scores ~0.94 ROC-AUC on the random split and collapses toward ~0.72 on the
contiguous (region-holdout) split, because a supervised model can only memorize the positions it
trained on. A zero-shot scorer has no training to overfit — the same score is read off a frozen
language model for every variant — so the hypothesis is that it holds its accuracy across all
three splits, including the hard contiguous one. This notebook tests that directly and reports it
next to the `02` numbers.

**Scorers** (project decisions D010–D013, D030–D031): the ESM ladder ESM-1b / ESM-1v /
ESM-2 {150M, 650M, 3B} / ESM-C {300M, 600M}, each under two scoring schemes — **masked-marginal**
(primary, D030) and **wildtype-marginal** (D031). Each `{model}__{scheme}` score is a surprisal
difference log P(mut) − log P(wt) at the mutated site. Equivalently this is **surprisal(wt) − surprisal(mut)** — the wild-type-referenced form of the ESM surprisal used in the earlier `ML EDA Notebook` (subtracting the WT surprisal cancels each position's baseline intolerance, isolating the effect of the specific substitution).

**Inputs read from disk** (never in-memory state, per CLAUDE.md):
- labels: `data/processed/traditional_ml_aa_identity/modeling_dataset.parquet`
- scores: `data/features/plm_masked_marginal/pllm_zeroshot_scores.parquet` — produced by the
  Colab twin `06_pllm_zeroshot_benchmark_colab.ipynb`; **run that first** and drop the parquet
  into place.

**Statistics** mirror `02` exactly: 5-fold random / modulo / contiguous splits, bootstrap CIs,
DeLong (ranking) and McNemar (thresholded) tests, and a dummy floor.

In [ ]:
# self-contained: resolve project root via .projectroot, read from disk
import sys, json, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'.projectroot').exists())
sys.path.insert(0, str(root)); from paths import *

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score,
    balanced_accuracy_score, f1_score, matthews_corrcoef, confusion_matrix, roc_curve,
    precision_recall_curve)

SEED = 42; N_SPLITS = 5; N_BOOT = 2000
AA = list("ACDEFGHIKLMNPQRSTVWY")

# the ESM ladder, in scale order within family; primary scheme first
MODEL_ORDER = ["esm1b","esm1v","esm2_150m","esm2_650m","esm2_3b","esmc_300m","esmc_600m"]
MODEL_LABEL = {"esm1b":"ESM-1b 650M","esm1v":"ESM-1v 650M","esm2_150m":"ESM-2 150M",
               "esm2_650m":"ESM-2 650M","esm2_3b":"ESM-2 3B","esmc_300m":"ESM-C 300M",
               "esmc_600m":"ESM-C 600M"}
SCHEMES = ["masked_marginal","wildtype_marginal"]
PRIMARY_SCHEME = "masked_marginal"   # D030

# project palette: blues, greens, purples, dark pinks only
MODEL_COLORS = {"esm1b":"#2c6fbb","esm1v":"#1f4e8c","esm2_150m":"#57a773","esm2_650m":"#2e8b57",
                "esm2_3b":"#1d6b3f","esmc_300m":"#7a4fa3","esmc_600m":"#b03060"}
PAL = {"blue":"#2c6fbb","green":"#2e8b57","purple":"#7a4fa3","pink":"#b03060","grey":"#9aa0a6"}
sns.set_theme(style="whitegrid")

NBNAME = "06_pllm_zeroshot_benchmark"
FIGDIR = FIGURES/NBNAME; FIGDIR.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)
print("root:", root, "| seed:", SEED)


## 1. Load labels + zero-shot scores, and validate the join

In [ ]:
df = pd.read_parquet(PROCESSED/"traditional_ml_aa_identity"/"modeling_dataset.parquet")
SCORES_PATH = FEATURES_PLM_MM/"pllm_zeroshot_scores.parquet"
assert SCORES_PATH.exists(), (
    f"scores not found at {SCORES_PATH}\n"
    "Run the Colab twin 06_pllm_zeroshot_benchmark_colab.ipynb first, then drop "
    "pllm_zeroshot_scores.parquet into data/features/plm_masked_marginal/.")
scores = pd.read_parquet(SCORES_PATH)

# --- boundary checks (CLAUDE.md): trust nothing that crossed from the Colab run ---
assert df.seq_id.is_unique and scores.seq_id.is_unique, "duplicate seq_id"
assert set(scores.seq_id) == set(df.seq_id), "score/label seq_id sets differ"
assert len(scores) == len(df), f"row count mismatch {len(scores)} vs {len(df)}"

# align scores to the label row order deterministically
data = df.merge(scores, on="seq_id", how="inner", validate="1:1")
data = data.sort_values(["position","mut_aa"]).reset_index(drop=True)
assert len(data) == len(df)

y = data.DMS_score_bin.values.astype(int)
pos = data.position.values
# which (model, scheme) score columns actually arrived (a model may have been skipped on Colab)
present = [(m,s) for m in MODEL_ORDER for s in SCHEMES if f"{m}__{s}" in data.columns]
score_cols = {f"{m}__{s}": data[f"{m}__{s}"].values.astype(float) for m,s in present}
assert score_cols, "no recognized {model}__{scheme} score columns present"
for c,v in score_cols.items():
    assert np.isfinite(v).all(), f"non-finite scores in {c}"

# --- label-leakage guard: a zero-shot score must NOT be a deterministic function of the label ---
# (it is computed with no access to y; assert the correlation is not implausibly perfect)
for c,v in score_cols.items():
    r = abs(np.corrcoef(v, y)[0,1])
    assert r < 0.99, f"{c} is near-perfectly correlated with the label (r={r:.3f}) — leakage?"

print(f"variants={len(data)} | class balance={np.bincount(y).tolist()} | "
      f"scorers present={len(score_cols)}")
print("present scorers:", list(score_cols.keys()))


## 2. Three split schemes (identical to `02`)

The same `random` / `modulo` / `contiguous` folds as the AA-identity benchmark, so the two
baselines are compared on exactly the same partitions. Zero-shot ranking (ROC/PR) does not depend
on training, but we still evaluate per fold to get split-wise means and variance — and the
**threshold** for the class-decision metrics is fit on the train fold only.

In [ ]:
def make_folds(data, n_splits=N_SPLITS, seed=SEED):
    idx = np.arange(len(data)); pos = data.position.values; folds={}
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    folds["random"] = list(skf.split(idx, data.DMS_score_bin.values))
    uniq = np.sort(data.position.unique())
    folds["modulo"] = [(idx[~np.isin(pos, uniq[uniq%n_splits==k])],
                        idx[ np.isin(pos, uniq[uniq%n_splits==k])]) for k in range(n_splits)]
    folds["contiguous"] = [(idx[~np.isin(pos,b)], idx[np.isin(pos,b)])
                           for b in np.array_split(uniq, n_splits)]
    return folds

folds = make_folds(data)
for scheme in ["modulo","contiguous"]:
    for k,(tr,te) in enumerate(folds[scheme]):
        assert not (set(pos[tr]) & set(pos[te])), f"{scheme} fold {k} leaks a position"
for scheme,fl in folds.items():
    tested=np.concatenate([te for _,te in fl]); assert len(np.unique(tested))==len(data)
    print(f"{scheme:11s} test sizes {[len(te) for _,te in fl]}")
print("no position leakage in modulo/contiguous; full coverage in all schemes")


## 3. Zero-shot evaluation

Each score is treated as a **fixed, untrained classifier**. Ranking metrics (ROC-AUC, PR-AUC) read
directly off the score. For the threshold-dependent metrics we pick a single decision threshold on
the **train fold** (Youden's J on the train ROC) and apply it to the held-out test fold — the only
thing "learned" is one scalar, and it never sees test labels. A most-frequent-class dummy gives the
no-skill floor, exactly as in `02`.

In [ ]:
UTIL = {"TP":1.0,"TN":1.0,"FN":-0.25,"FP":-1.0}   # same utility weights as 02
def utility(y_true,y_pred):
    tn,fp,fn,tp = confusion_matrix(y_true,y_pred,labels=[0,1]).ravel()
    return (UTIL["TP"]*tp+UTIL["TN"]*tn+UTIL["FN"]*fn+UTIL["FP"]*fp)/len(y_true)

def fit_threshold(s_tr, y_tr):
    # Youden's J on the train ROC; higher score => positive (functional) class
    fpr,tpr,thr = roc_curve(y_tr, s_tr)
    j = np.argmax(tpr - fpr)
    return thr[j]

def metrics_at(y_true, s, thr):
    pred = (s >= thr).astype(int)
    return dict(roc_auc=roc_auc_score(y_true,s), pr_auc=average_precision_score(y_true,s),
        accuracy=accuracy_score(y_true,pred), balanced_acc=balanced_accuracy_score(y_true,pred),
        f1=f1_score(y_true,pred), mcc=matthews_corrcoef(y_true,pred), utility=utility(y_true,pred))

def run_zeroshot(score_cols, y, folds):
    oof_score, oof_pred, fold_metrics, thr_used = {}, {}, {}, {}
    for scheme, fl in folds.items():
        # real scorers
        for name, s in score_cols.items():
            key=(scheme,name); pred=np.full(len(y), -1); per=[]; thrs=[]
            for tr,te in fl:
                thr = fit_threshold(s[tr], y[tr]); thrs.append(thr)
                pred[te] = (s[te]>=thr).astype(int)
                per.append(metrics_at(y[te], s[te], thr))
            oof_score[key]=s                       # raw score is split-invariant (no training)
            oof_pred[key]=pred; fold_metrics[key]=pd.DataFrame(per); thr_used[key]=float(np.median(thrs))
        # dummy: most-frequent class, constant score
        maj = int(np.bincount(y).argmax()); key=(scheme,"dummy")
        s = np.full(len(y), float(maj)); pred=np.full(len(y), maj); per=[]
        for tr,te in fl:
            per.append(metrics_at(y[te], s[te]+np.zeros_like(s[te]), 0.5))
        oof_score[key]=s; oof_pred[key]=pred; fold_metrics[key]=pd.DataFrame(per); thr_used[key]=0.5
    return oof_score, oof_pred, fold_metrics, thr_used

t0=time.time()
oof_score, oof_pred, fold_metrics, thr_used = run_zeroshot(score_cols, y, folds)
print(f"evaluated {len(fold_metrics)} (scorer x split) combos in {time.time()-t0:.1f}s")


In [ ]:
# assemble the mean +/- std metrics grid (same shape as 02_...metrics_grid.csv)
METRIC_KEYS=["roc_auc","pr_auc","accuracy","balanced_acc","f1","mcc","utility"]
rows=[]
for (scheme,name),fm in fold_metrics.items():
    r={"split":scheme,"scorer":name}
    if name!="dummy":
        m,sch=name.rsplit("__",1); r["model"]=m; r["scheme"]=sch
    else:
        r["model"]="dummy"; r["scheme"]="-"
    for k in METRIC_KEYS:
        r[f"{k}_mean"]=round(fm[k].mean(),4); r[f"{k}_std"]=round(fm[k].std(),4)
    rows.append(r)
grid=pd.DataFrame(rows).sort_values(["split","roc_auc_mean"],ascending=[True,False]).reset_index(drop=True)
# headline view: primary scheme only, ranking metric
prim=grid[(grid.scheme==PRIMARY_SCHEME)|(grid.model=="dummy")]
print("=== ROC-AUC (mean+/-std) by split, primary scheme (masked-marginal) ===")
print(prim.pivot_table(index="model",columns="split",values="roc_auc_mean").reindex(MODEL_ORDER+["dummy"]).round(3).to_string())


## 4. Statistical significance (mirrors `02`)

Bootstrap 95% CIs on the pooled out-of-fold predictions, DeLong tests comparing each model's
ranking against the best model per split, and McNemar tests on the thresholded predictions. Note
that for a zero-shot score the **ranking** (and therefore ROC-AUC, PR-AUC, DeLong) is
split-invariant — the raw score does not change with the fold structure — so any split differences
you see in ranking metrics come only from which variants land in each test fold; the
threshold-dependent metrics (accuracy, F1, MCC, utility, McNemar) do vary with the per-fold
threshold.

In [ ]:
def bootstrap_ci(y_true, proba, fn, n_boot=N_BOOT, seed=SEED):
    rng=np.random.default_rng(seed); n=len(y_true); vals=[]
    for _ in range(n_boot):
        idx=rng.integers(0,n,n)
        if len(np.unique(y_true[idx]))<2: continue
        vals.append(fn(y_true[idx], proba[idx]))
    return float(np.mean(vals)), float(np.percentile(vals,2.5)), float(np.percentile(vals,97.5))

def _midrank(x):
    J=np.argsort(x); Z=x[J]; N=len(x); T=np.zeros(N); i=0
    while i<N:
        j=i
        while j<N and Z[j]==Z[i]: j+=1
        T[i:j]=0.5*(i+j-1)+1; i=j
    T2=np.empty(N); T2[J]=T; return T2

def delong_test(y_true, p1, p2):
    y=y_true.astype(int); posm=y==1; negm=y==0; m=posm.sum(); n=negm.sum()
    preds=np.vstack([p1,p2]); tz=np.array([_midrank(p) for p in preds])
    auc=np.array([(stats.rankdata(np.concatenate([p[posm],p[negm]]))[:m].sum()-m*(m+1)/2)/(m*n) for p in preds])
    v01=(tz[:,y==1]-np.array([_midrank(p[posm]) for p in preds]))/n
    v10=1.0-(tz[:,y==0]-np.array([_midrank(p[negm]) for p in preds]))/m
    sx=np.cov(v01); sy=np.cov(v10); s=sx/m+sy/n
    var=s[0,0]+s[1,1]-2*s[0,1]
    if var<=0: return auc[0],auc[1],1.0
    z=(auc[0]-auc[1])/np.sqrt(var); return float(auc[0]),float(auc[1]),float(2*stats.norm.sf(abs(z)))

from statsmodels.stats.contingency_tables import mcnemar
def mcnemar_test(y_true, pred1, pred2):
    a=pred1.astype(int); b=pred2.astype(int)
    n01=((a==y_true)&(b!=y_true)).sum(); n10=((a!=y_true)&(b==y_true)).sum()
    r=mcnemar([[0,int(n01)],[int(n10),0]], exact=False, correction=True)
    return int(n01), int(n10), float(r.pvalue)
print("significance helpers ready")


In [ ]:
# CIs on pooled OOF (ranking) + significance vs best model per split, and vs dummy
real=list(score_cols.keys())
ci_rows=[]; sig_rows=[]
for scheme in folds:
    aucs={nm: roc_auc_score(y, oof_score[(scheme,nm)]) for nm in real}
    best=max(aucs, key=aucs.get)
    for nm in real+["dummy"]:
        s=oof_score[(scheme,nm)]
        ra,rlo,rhi=bootstrap_ci(y,s,roc_auc_score)
        pa,plo,phi=bootstrap_ci(y,s,average_precision_score)
        ci_rows.append(dict(split=scheme,scorer=nm,roc_auc=ra,roc_lo=rlo,roc_hi=rhi,
                            pr_auc=pa,pr_lo=plo,pr_hi=phi))
    for nm in real:
        if nm==best:
            sig_rows.append(dict(split=scheme,scorer=nm,vs_best=best,delong_p=np.nan,
                                 mcnemar_p=np.nan,vs_dummy_delong_p=np.nan,note="best")); continue
        _,_,dp=delong_test(y, oof_score[(scheme,nm)], oof_score[(scheme,best)])
        _,_,mp=mcnemar_test(y, oof_pred[(scheme,nm)], oof_pred[(scheme,best)])
        _,_,dpd=delong_test(y, oof_score[(scheme,nm)], oof_score[(scheme,"dummy")])
        sig_rows.append(dict(split=scheme,scorer=nm,vs_best=best,delong_p=dp,mcnemar_p=mp,
                             vs_dummy_delong_p=dpd,note=""))
ci=pd.DataFrame(ci_rows); sig=pd.DataFrame(sig_rows)
print("best scorer per split:", {s: max({nm:roc_auc_score(y,oof_score[(s,nm)]) for nm in real}, key=lambda k:roc_auc_score(y,oof_score[(s,k)])) for s in folds})
display(ci[ci.scorer.str.contains(PRIMARY_SCHEME)|(ci.scorer=="dummy")].round(3))


## 5. Split-robustness and cross-reference to the `02` AA-identity baseline

The headline comparison. For each split, take the best zero-shot scorer and the best `02`
supervised model, and line up their ROC-AUC. The AA-identity model's drop from random →
contiguous is its position-memorization gap; the zero-shot scorer's flat line across splits is the
robustness the pLLM approach is meant to buy.

In [ ]:
AA_GRID = TABLES/"02_traditional_ml_aa_identity_benchmark_metrics_grid.csv"
xref_rows=[]
aa=pd.read_csv(AA_GRID) if AA_GRID.exists() else None
for scheme in ["random","modulo","contiguous"]:
    zs={nm: roc_auc_score(y, oof_score[(scheme,nm)]) for nm in real}
    zbest=max(zs,key=zs.get)
    row=dict(split=scheme, zeroshot_best_model=zbest, zeroshot_best_roc_auc=round(zs[zbest],4))
    if aa is not None:
        a=aa[(aa.split==scheme)&(aa.model!="dummy")].sort_values("roc_auc_mean",ascending=False)
        if len(a):
            row.update(aa_identity_best_model=a.iloc[0].model,
                       aa_identity_best_roc_auc=round(float(a.iloc[0].roc_auc_mean),4),
                       delta_zeroshot_minus_aa=round(zs[zbest]-float(a.iloc[0].roc_auc_mean),4))
    xref_rows.append(row)
xref=pd.DataFrame(xref_rows)
display(xref)
if aa is not None:
    drop_aa = xref.aa_identity_best_roc_auc.iloc[0]-xref.aa_identity_best_roc_auc.iloc[2]
    drop_zs = xref.zeroshot_best_roc_auc.iloc[0]-xref.zeroshot_best_roc_auc.iloc[2]
    print(f"random->contiguous ROC-AUC drop: AA-identity {drop_aa:+.3f} vs zero-shot {drop_zs:+.3f}")
else:
    print("NOTE: 02 metrics grid not found — cross-reference columns omitted.")


In [ ]:
# 5d. consolidated AA-identity vs zero-shot comparison (all metrics, best model per arm per split)
# Two baselines side by side: AA-identity supervised (02 -- raw amino acids only, no language
# model) vs the D027 no-training control (zero-shot pLLM). D027's designated no-language-model
# control is a physicochemical-descriptor model (separate, not yet built); AA-identity is the
# supervised no-LM counterpart available now. "Best" = highest-ROC-AUC model within each
# arm on each split. Delta = zeroshot - AA-identity (positive => pLLM wins).
METR=[("roc_auc","ROC-AUC"),("pr_auc","PR-AUC"),("balanced_acc","bal.acc"),
      ("mcc","MCC"),("utility","utility")]
cmp_rows=[]
for scheme in ["random","modulo","contiguous"]:
    # zero-shot best (primary scheme), by ROC-AUC
    zs=grid[(grid.scheme==PRIMARY_SCHEME)&(grid.split==scheme)].sort_values("roc_auc_mean",ascending=False)
    zb=zs.iloc[0]
    if aa is not None:
        ab=aa[(aa.split==scheme)&(aa.model!="dummy")].sort_values("roc_auc_mean",ascending=False).iloc[0]
    for key,lab in METR:
        row=dict(split=scheme, metric=lab,
                 zeroshot=round(float(zb[f"{key}_mean"]),4),
                 zeroshot_model=MODEL_LABEL.get(zb.model,zb.model))
        if aa is not None:
            row.update(aa_identity=round(float(ab[f"{key}_mean"]),4),
                       aa_identity_model=ab.model,
                       delta=round(float(zb[f"{key}_mean"])-float(ab[f"{key}_mean"]),4))
        cmp_rows.append(row)
aa_vs_zs=pd.DataFrame(cmp_rows)
display(aa_vs_zs)
if aa is not None:
    wins=(aa_vs_zs.delta>0).sum()
    print(f"zero-shot beats AA-identity in {wins}/{len(aa_vs_zs)} (metric x split) cells; "
          f"on the region-holdout splits (modulo+contiguous): "
          f"{(aa_vs_zs[aa_vs_zs.split!='random'].delta>0).sum()}/{(aa_vs_zs.split!='random').sum()}")


In [ ]:
# scheme comparison: masked-marginal vs wildtype-marginal, per model (pooled ROC-AUC)
sc_rows=[]
for m in MODEL_ORDER:
    row={"model":m}
    for sch in SCHEMES:
        nm=f"{m}__{sch}"
        if nm in score_cols: row[sch]=round(roc_auc_score(y,score_cols[nm]),4)
    if len(row)>1: sc_rows.append(row)
scheme_cmp=pd.DataFrame(sc_rows)
if {"masked_marginal","wildtype_marginal"}.issubset(scheme_cmp.columns):
    scheme_cmp["delta_mm_minus_wm"]=(scheme_cmp.masked_marginal-scheme_cmp.wildtype_marginal).round(4)
display(scheme_cmp)


## 6. Figures

In [ ]:
# 6a. ROC curves — one panel per split, one line per model (primary scheme)
present_models=[m for m in MODEL_ORDER if f"{m}__{PRIMARY_SCHEME}" in score_cols]
fig,axes=plt.subplots(1,3,figsize=(15,4.6),sharey=True)
for ax,scheme in zip(axes,["random","modulo","contiguous"]):
    for m in present_models:
        s=score_cols[f"{m}__{PRIMARY_SCHEME}"]; fpr,tpr,_=roc_curve(y,s)
        ax.plot(fpr,tpr,color=MODEL_COLORS[m],lw=1.6,
                label=f"{MODEL_LABEL[m]} ({roc_auc_score(y,s):.3f})")
    ax.plot([0,1],[0,1],ls="--",color=PAL["grey"],lw=1)
    ax.set_title(f"{scheme} split"); ax.set_xlabel("false positive rate")
    ax.legend(fontsize=7,loc="lower right",frameon=False)
axes[0].set_ylabel("true positive rate")
fig.suptitle("Zero-shot ROC by split (masked-marginal)",y=1.02)
fig.tight_layout(); fig.savefig(FIGDIR/"roc_curves.pdf",bbox_inches="tight"); fig.savefig(FIGDIR/"roc_curves.png",bbox_inches="tight", dpi=200); plt.show()


In [ ]:
# 6b. PR curves
fig,axes=plt.subplots(1,3,figsize=(15,4.6),sharey=True)
base=y.mean()
for ax,scheme in zip(axes,["random","modulo","contiguous"]):
    for m in present_models:
        s=score_cols[f"{m}__{PRIMARY_SCHEME}"]; pr,rc,_=precision_recall_curve(y,s)
        ax.plot(rc,pr,color=MODEL_COLORS[m],lw=1.6,
                label=f"{MODEL_LABEL[m]} ({average_precision_score(y,s):.3f})")
    ax.axhline(base,ls="--",color=PAL["grey"],lw=1)
    ax.set_title(f"{scheme} split"); ax.set_xlabel("recall")
    ax.legend(fontsize=7,loc="lower left",frameon=False)
axes[0].set_ylabel("precision")
fig.suptitle("Zero-shot precision-recall by split (masked-marginal)",y=1.02)
fig.tight_layout(); fig.savefig(FIGDIR/"pr_curves.pdf",bbox_inches="tight"); fig.savefig(FIGDIR/"pr_curves.png",bbox_inches="tight", dpi=200); plt.show()


In [ ]:
# 6c. metric bars with bootstrap CIs (ROC-AUC), grouped by split
cim=ci[ci.scorer.str.endswith(PRIMARY_SCHEME)].copy()
cim["model"]=cim.scorer.str.replace(f"__{PRIMARY_SCHEME}","",regex=False)
schemes=["random","modulo","contiguous"]
fig,ax=plt.subplots(figsize=(12,5)); w=0.11
x=np.arange(len(schemes))
for i,m in enumerate(present_models):
    sub=cim[cim.model==m].set_index("split").reindex(schemes)
    err=np.vstack([sub.roc_auc-sub.roc_lo, sub.roc_hi-sub.roc_auc])
    ax.bar(x+i*w, sub.roc_auc, w, yerr=err, capsize=2, color=MODEL_COLORS[m],
           label=MODEL_LABEL[m], edgecolor="black", linewidth=.4)
ax.axhline(0.5,ls="--",color=PAL["grey"],lw=1,label="no-skill")
ax.set_xticks(x+w*(len(present_models)-1)/2); ax.set_xticklabels(schemes)
ax.set_ylabel("ROC-AUC (95% bootstrap CI)"); ax.set_ylim(0.4,1.0)
ax.set_title("Zero-shot ROC-AUC by split and model (masked-marginal)")
ax.legend(fontsize=8,ncol=2,frameon=False)
fig.tight_layout(); fig.savefig(FIGDIR/"metric_bars.pdf",bbox_inches="tight"); fig.savefig(FIGDIR/"metric_bars.png",bbox_inches="tight", dpi=200); plt.show()


In [ ]:
# 6d. utility-score bar chart (mean utility per prediction), grouped by split -- mirrors 02 (8e)
# same weights as 02: TP +1, TN +1, FN -0.25, FP -1, normalized per prediction; threshold fit
# on train fold only (Youden's J). dummy = majority-class constant prediction.
gu=grid[(grid.scheme==PRIMARY_SCHEME)|(grid.model=="dummy")].copy()
schemes=["random","modulo","contiguous"]
fig,ax=plt.subplots(figsize=(12,4.8)); w=0.11
x=np.arange(len(schemes))
for i,m in enumerate(present_models):
    sub=gu[gu.model==m].set_index("split").reindex(schemes)
    ax.bar(x+i*w, sub.utility_mean, w, yerr=sub.utility_std, capsize=2,
           color=MODEL_COLORS[m], label=MODEL_LABEL[m], edgecolor="black", linewidth=.4)
dv=[gu[(gu.split==s)&(gu.model=="dummy")].utility_mean.values[0] for s in schemes]
ax.plot(x+w*(len(present_models)-1)/2, dv, "k_", ms=18, label="dummy")
ax.axhline(0,ls=":",color=PAL["grey"],lw=1)
ax.set_xticks(x+w*(len(present_models)-1)/2); ax.set_xticklabels(schemes)
ax.set_ylabel("mean utility / prediction"); ax.set_title("Utility score (TP+1, TN+1, FN-0.25, FP-1)")
ax.legend(fontsize=8,ncol=2,frameon=False)
fig.tight_layout(); fig.savefig(FIGDIR/"utility_bars.pdf",bbox_inches="tight"); fig.savefig(FIGDIR/"utility_bars.png",bbox_inches="tight", dpi=200); plt.show()


In [ ]:
# 6d. split-robustness contrast: zero-shot best vs AA-identity best, ROC-AUC by split
fig,ax=plt.subplots(figsize=(7.5,5))
schemes=["random","modulo","contiguous"]
zbest_line=[max(roc_auc_score(y,oof_score[(s,nm)]) for nm in real) for s in schemes]
ax.plot(schemes, zbest_line, "-o", color=PAL["purple"], lw=2.2, ms=8, label="zero-shot pLLM (best)")
if aa is not None:
    aabest=[float(aa[(aa.split==s)&(aa.model!="dummy")].roc_auc_mean.max()) for s in schemes]
    ax.plot(schemes, aabest, "-s", color=PAL["pink"], lw=2.2, ms=8, label="AA-identity supervised (best, 02)")
ax.axhline(0.5,ls="--",color=PAL["grey"],lw=1,label="no-skill")
ax.set_ylabel("ROC-AUC"); ax.set_ylim(0.45,1.0)
ax.set_title("Split robustness: does accuracy survive region holdout?")
ax.legend(frameon=False)
fig.tight_layout(); fig.savefig(FIGDIR/"split_robustness.pdf",bbox_inches="tight"); fig.savefig(FIGDIR/"split_robustness.png",bbox_inches="tight", dpi=200); plt.show()


In [ ]:
# 6e. consolidated comparison figure: AA-identity vs zero-shot, ROC-AUC across splits
# (headline metric; the full-metric numbers live in the comparison table above)
if aa is not None:
    fig,ax=plt.subplots(figsize=(8,4.8)); schemes=["random","modulo","contiguous"]
    x=np.arange(len(schemes)); w=0.36
    zvals=[float(grid[(grid.scheme==PRIMARY_SCHEME)&(grid.split==s)].roc_auc_mean.max()) for s in schemes]
    avals=[float(aa[(aa.split==s)&(aa.model!="dummy")].roc_auc_mean.max()) for s in schemes]
    ax.bar(x-w/2, avals, w, color=PAL["pink"],   edgecolor="black", linewidth=.4, label="AA-identity supervised (02)")
    ax.bar(x+w/2, zvals, w, color=PAL["purple"], edgecolor="black", linewidth=.4, label="zero-shot pLLM (best)")
    for xi,(a,z) in enumerate(zip(avals,zvals)):
        ax.text(xi-w/2,a+.008,f"{a:.3f}",ha="center",fontsize=8)
        ax.text(xi+w/2,z+.008,f"{z:.3f}",ha="center",fontsize=8)
    ax.axhline(0.5,ls="--",color=PAL["grey"],lw=1,label="no-skill")
    ax.set_xticks(x); ax.set_xticklabels(schemes)
    ax.set_ylabel("ROC-AUC"); ax.set_ylim(0.45,1.0)
    ax.set_title("Two bounding baselines: AA-identity (supervised) vs zero-shot pLLM")
    ax.legend(frameon=False,fontsize=8)
    fig.tight_layout(); fig.savefig(FIGDIR/"aa_vs_zeroshot_comparison.pdf",bbox_inches="tight")
    fig.savefig(FIGDIR/"aa_vs_zeroshot_comparison.png",bbox_inches="tight", dpi=200); plt.show()
else:
    print("02 grid not found — consolidated comparison figure skipped")


In [ ]:
# 6e. scheme comparison: masked-marginal vs wildtype-marginal per model (pooled ROC-AUC)
if {"masked_marginal","wildtype_marginal"}.issubset(scheme_cmp.columns):
    fig,ax=plt.subplots(figsize=(9,5)); x=np.arange(len(scheme_cmp)); w=0.38
    ax.bar(x-w/2, scheme_cmp.masked_marginal, w, color=PAL["blue"], label="masked-marginal",
           edgecolor="black", linewidth=.4)
    ax.bar(x+w/2, scheme_cmp.wildtype_marginal, w, color=PAL["green"], label="wildtype-marginal",
           edgecolor="black", linewidth=.4)
    ax.axhline(0.5,ls="--",color=PAL["grey"],lw=1)
    ax.set_xticks(x); ax.set_xticklabels([MODEL_LABEL[m] for m in scheme_cmp.model], rotation=30, ha="right")
    ax.set_ylabel("pooled ROC-AUC"); ax.set_ylim(0.4,1.0)
    ax.set_title("Scoring-scheme comparison (D030 vs D031)")
    ax.legend(frameon=False)
    fig.tight_layout(); fig.savefig(FIGDIR/"scheme_comparison.pdf",bbox_inches="tight"); fig.savefig(FIGDIR/"scheme_comparison.png",bbox_inches="tight", dpi=200); plt.show()
else:
    print("only one scheme present — scheme_comparison figure skipped")


## 7. Save tables

In [ ]:
grid.to_csv(TABLES/f"{NBNAME}_metrics_grid.csv", index=False)
ci.to_csv(TABLES/f"{NBNAME}_bootstrap_ci.csv", index=False)
sig.to_csv(TABLES/f"{NBNAME}_significance.csv", index=False)
xref.to_csv(TABLES/f"{NBNAME}_vs_aa_identity.csv", index=False)
aa_vs_zs.to_csv(TABLES/f"{NBNAME}_aa_vs_zeroshot_comparison.csv", index=False)
scheme_cmp.to_csv(TABLES/f"{NBNAME}_scheme_comparison.csv", index=False)
pd.DataFrame([{"scorer":k[1],"split":k[0],"threshold":v} for k,v in thr_used.items()]
             ).to_csv(TABLES/f"{NBNAME}_thresholds.csv", index=False)
print("saved tables:")
for f in sorted(TABLES.glob(f"{NBNAME}_*.csv")): print(" ", f.name)
print("saved figures:")
for f in sorted(FIGDIR.glob("*.pdf")): print(" ", f.name)


## 8. Summary

Read the numbers off the tables above; the interpretation to state next to them:

- **Does zero-shot beat the dummy floor?** Every ESM scorer's ROC-AUC vs the 0.5 no-skill line,
  with DeLong significance in `..._significance.csv`.
- **Does it hold across splits?** The `split_robustness` figure and `..._vs_aa_identity.csv` are the
  headline: a flat zero-shot line across random/modulo/contiguous, contrasted with the AA-identity
  model's collapse on the contiguous (region-holdout) split from `02`.
- **Scale and scheme effects.** The ESM ladder (1b → 1v → 2 → C) in the metric bars, and
  masked-marginal vs wildtype-marginal in the scheme comparison (D030/D031).

**Limitation, stated where the number lives (D039).** These surprisal scores are shaped heavily by
foldability and stability, so they can under-detect a *catalytic-but-stable* knockout — a mutation
that abolishes activity without destabilizing the fold. A zero-shot scorer that looks strong here
may still miss exactly the active-site variants (S70, K73, S130, E166) that matter clinically; the
structure-based features and the wet-lab panel are what test that blind spot. This is a
**no-training control**, not the final predictor.